In [16]:
# Muat data dan lakukan pemeriksaan awal
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = Path("../data/harga_komoditas.csv")
assert DATA_PATH.exists(), f"File tidak ditemukan: {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH, parse_dates=["tanggal"])
df = df.sort_values(["komoditas", "tanggal"]).reset_index(drop=True)
df["bulan"] = df["tanggal"].dt.to_period("M")

print(f"Baris          : {len(df):,}")
print(f"Komoditas      : {df['komoditas'].nunique()}")
print(f"Rentang tanggal: {df['tanggal'].min().date()} s.d. {df['tanggal'].max().date()}")
print(f"Jumlah bulan   : {df['bulan'].nunique()}")
print(f"Duplikat       : {df.duplicated(['tanggal', 'komoditas']).sum()}")
print("\nNilai kosong per kolom:")
print(df.isna().sum().to_string())
print("\nTipe data:")
print(df.dtypes.to_string())

df.head()

Baris          : 85,336
Komoditas      : 71
Rentang tanggal: 2023-01-01 s.d. 2026-08-31
Jumlah bulan   : 44
Duplikat       : 0

Nilai kosong per kolom:
tanggal      0
komoditas    0
satuan       0
harga        0
bulan        0

Tipe data:
tanggal      datetime64[us]
komoditas               str
satuan                  str
harga               float64
bulan             period[M]


,tanggal,komoditas,satuan,harga,bulan
0,2023-01-01,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.0,2023-01
1,2023-01-02,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.0,2023-01
2,2023-01-03,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.0,2023-01
3,2023-01-04,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.0,2023-01
4,2023-01-05,Air Mineral (Aqua Galon),19 Ltr/Galon,18000.0,2023-01


In [17]:
# daftar komoditas

daftar = (
    df.groupby("komoditas")
    .agg(
        satuan=("satuan", lambda s: " | ".join(sorted(s.dropna().unique()))),
        tanggal_awal=("tanggal", "min"),
        tanggal_akhir=("tanggal", "max"),
        hari_tercatat=("tanggal", "nunique"),
        bulan_tercatat=("bulan", "nunique"),
        harga_min=("harga", "min"),
        harga_rata2=("harga", "mean"),
        harga_maks=("harga", "max"),
        harga_terakhir=("harga", "last"),
    )
    .reset_index()
)

kolom_harga = ["harga_min", "harga_rata2", "harga_maks", "harga_terakhir"]
daftar[kolom_harga] = daftar[kolom_harga].round(0)
daftar["tanggal_awal"] = daftar["tanggal_awal"].dt.date
daftar["tanggal_akhir"] = daftar["tanggal_akhir"].dt.date
daftar.index = daftar.index + 1   # nomor urut mulai dari 1

# Simpan untuk dokumentasi
Path("../reports/tables").mkdir(parents=True, exist_ok=True)
daftar.to_csv("../reports/tables/daftar_komoditas.csv", index_label="no")

print(f"Total komoditas: {len(daftar)}")
daftar

Total komoditas: 71


,komoditas,satuan,tanggal_awal,tanggal_akhir,hari_tercatat,bulan_tercatat,harga_min,harga_rata2,harga_maks,harga_terakhir
1,Air Mineral (Aqua Galon),19 Ltr/Galon,2023-01-01,2026-08-31,1333,44,17750.0,18509.0,19333.0,19000.0
2,Apel,Kg,2024-03-27,2026-08-31,884,30,20000.0,30149.0,36500.0,33667.0
3,Bawang Bombay,Kg,2023-01-01,2026-08-31,1335,44,21000.0,31636.0,45000.0,31429.0
4,Bawang Merah (Kualitas Lokal),Kg,2023-01-01,2026-08-31,1335,44,18714.0,32870.0,50000.0,30000.0
5,Bawang Putih (Jenis Kating),Kg,2023-01-01,2026-08-31,1335,44,26143.0,37215.0,50000.0,36571.0
6,Bawang Putih (Jenis Sinco),Kg,2023-01-01,2026-08-31,1335,44,19143.0,33434.0,44857.0,31286.0
7,Bayam (Jenis Sayur),Ikat Kecil,2023-01-01,2026-08-31,1335,44,1500.0,2053.0,4000.0,2250.0
8,Beras Jagung,Kg,2023-01-01,2026-08-31,1335,44,9500.0,11102.0,12000.0,11667.0
9,Beras Ketan (Jenis Siyem),Kg,2023-01-01,2026-08-31,1335,44,17000.0,21114.0,27200.0,23167.0
10,Beras Medium Bulog,Kg,2023-01-01,2026-08-31,1092,40,8933.0,11532.0,13500.0,12300.0


In [18]:
# harga rata-rata per bulan per komoditas
pivot_bulanan = df.pivot_table(
    index="komoditas", columns="bulan", values="harga", aggfunc="mean"
).round(0)
pivot_bulanan.columns = pivot_bulanan.columns.astype(str)

print(f"Ukuran: {pivot_bulanan.shape[0]} komoditas x {pivot_bulanan.shape[1]} bulan")
pivot_bulanan

Ukuran: 71 komoditas x 44 bulan


bulan,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08
komoditas,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Air Mineral (Aqua Galon),18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,17942.0,17750.0,17750.0,17750.0,17968.0,18000.0,18000.0,18000.0,18000.0,18000.0,18000.0,18210.0,18500.0,18500.0,18667.0,18857.0,19000.0,19000.0,19000.0,19000.0,19000.0,19000.0,19000.0,19132.0,19250.0,19296.0,19333.0,19333.0,19333.0,19333.0,19333.0,19144.0,19054.0,19118.0
Apel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34700.0,32172.0,30148.0,30167.0,28019.0,31587.0,31267.0,29387.0,28933.0,28142.0,27645.0,27071.0,27667.0,28500.0,28323.0,28600.0,28323.0,29032.0,29600.0,29232.0,29933.0,31766.0,30750.0,30750.0,32444.0,33250.0,33250.0,31733.0,33382.0,32567.0
Bawang Bombay,21816.0,21490.0,23267.0,34386.0,41138.0,38233.0,30525.0,28571.0,27438.0,27065.0,26839.0,26167.0,26737.0,27481.0,29286.0,30857.0,40369.0,39938.0,36424.0,32889.0,32652.0,31470.0,30238.0,31318.0,30687.0,30051.0,30105.0,32490.0,35207.0,35571.0,37687.0,33705.0,32633.0,33253.0,32405.0,32507.0,32963.0,32500.0,33018.0,32905.0,32816.0,31786.0,31412.0,30925.0
Bawang Merah (Kualitas Lokal),34369.0,34388.0,33115.0,35176.0,38184.0,34193.0,29806.0,23009.0,20643.0,19419.0,23295.0,27235.0,28576.0,25074.0,25336.0,40512.0,44263.0,35105.0,24286.0,20737.0,22086.0,23065.0,31052.0,34475.0,31834.0,27980.0,38719.0,42181.0,34106.0,36086.0,39719.0,42917.0,32824.0,36180.0,38114.0,41341.0,33622.0,36490.0,37184.0,37254.0,41373.0,46029.0,35336.0,30184.0
Bawang Putih (Jenis Kating),27673.0,30577.0,33871.0,33671.0,35871.0,34986.0,37535.0,35751.0,34952.0,34410.0,34238.0,35429.0,36645.0,36877.0,38493.0,42924.0,44009.0,41476.0,38714.0,38484.0,37624.0,38493.0,39305.0,40779.0,41274.0,41934.0,42362.0,44467.0,42576.0,41695.0,39286.0,36032.0,34371.0,34433.0,34867.0,35111.0,36120.0,35490.0,35452.0,35376.0,34018.0,35210.0,37645.0,36687.0
Bawang Putih (Jenis Sinco),24406.0,25959.0,28277.0,29162.0,32332.0,32067.0,33968.0,33544.0,32586.0,32152.0,31705.0,32332.0,33111.0,32823.0,34576.0,38038.0,38783.0,36871.0,35908.0,35456.0,35173.0,35559.0,36029.0,37622.0,37945.0,38219.0,38795.0,40405.0,37396.0,35105.0,33180.0,30982.0,29481.0,30484.0,31114.0,32530.0,32562.0,32122.0,32355.0,31413.0,30332.0,32562.0,33553.0,31788.0
Bayam (Jenis Sayur),1648.0,1782.0,1912.0,1900.0,1917.0,1867.0,1605.0,1619.0,1650.0,1689.0,1705.0,1758.0,1721.0,1771.0,1786.0,1786.0,1824.0,1833.0,1769.0,1710.0,1625.0,1728.0,1828.0,1852.0,1981.0,2077.0,1956.0,2222.0,2161.0,2250.0,2374.0,2237.0,2078.0,2108.0,2089.0,2339.0,2298.0,2250.0,2323.0,2355.0,3823.0,3772.0,3048.0,2301.0
Beras Jagung,10000.0,9720.0,10316.0,10600.0,10600.0,10440.0,10187.0,10102.0,10040.0,10400.0,10333.0,10400.0,10389.0,10667.0,11332.0,11656.0,11642.0,11403.0,11333.0,11290.0,11333.0,11366.0,11333.0,11382.0,11500.0,11500.0,11500.0,11500.0,11452.0,11500.0,11452.0,11591.0,11511.0,11544.0,11460.0,11498.0,11429.0,11429.0,11429.0,11429.0,11429.0,11433.0,11809.0,11800.0
Beras Ketan (Jenis Siyem),19250.0,19777.0,20148.0,20907.0,20174.0,19653.0,18806.0,18671.0,18453.0,18400.0,18262.0,18325.0,18269.0,19000.0,19339.0,20578.0,20683.0,20756.0,20833.0,20833.0,20833.0,20833.0,20833.0,21124.0,21833.0,21833.0,21833.0,21833.0,21677.0,21833.0,21677.0,22414.0,22483.0,22327.0,22967.0,23000.0,23000.0,23000.0,23000.0,23450.0,23415.0,25231.0,24119.0,23653.0


In [19]:
# Pengecualian 22 komoditas sesuai kesepakatan
PENGECUALIAN = {
    "Beras Premium": "Beras Premium",
    "Beras Medium Bulog": "Beras Medium Bulog",
    "Beras Medium Non Bulog": "Beras Medium Non Bulog",
    "Gula Pasir Lokal Curah": "Gula Pasir Lokal Curah",
    "Minyak Goreng Curah": "Minyak Goreng Curah (Bening)",
    "Minyak KITA": "Minyak Kita",
    "Daging Sapi Grade Atas": "Daging Sapi Grade Atas",
    "Daging Sapi Grade Bawah": "Daging Sapi Grade Bawah",
    "Daging Sapi Grade Sedang": "Daging Sapi Grade Sedang",
    "Daging Ayam Broiler/Ras": "Daging Ayam Broiler",
    "Telur Ayam Ras": "Telur Ayam Horn",
    "Ikan Tongkol Segar": "Ikan Tongkol Segar (Uk. Sedang)",
    "Ikan Mujair Segar": "Ikan Mujair Segar (Uk. Sedang)",
    "Ikan Lele Segar": "Ikan Lele Segar (Ukuran Sedang)",
    "Ikan Bandeng Segar": "Ikan Bandeng Segar (Uk. Sedang)",
    'Tepung Terigu "Segitiga Biru" Curah': 'Tepung Terigu "Segitiga Biru" Curah',
    "Cabe Kecil Rawit": "Cabe Merah Kecil-Rawit",
    "Cabe Merah Besar": "Cabe Merah Besar",
    "Cabe Merah Keriting": "Cabe Merah Kriting",
    "Bawang Putih (Jenis Kating)": "Bawang Putih (Jenis Kating)",
    "Bawang Putih (Jenis Sinco)": "Bawang Putih (Jenis Sinco)",
    "Bawang Merah (Kualitas Lokal)": "Bawang Merah (Kualitas Lokal)",
}

semua_nama = set(df["komoditas"].unique())
tidak_cocok = {k: v for k, v in PENGECUALIAN.items() if v not in semua_nama}
if tidak_cocok:
    raise ValueError(f"Nama berikut tidak ada di dataset: {tidak_cocok}")

EXCLUDED = set(PENGECUALIAN.values())
print(f"Dikecualikan : {len(EXCLUDED)} komoditas")
print(f"Tersisa      : {len(semua_nama - EXCLUDED)} komoditas")

Dikecualikan : 22 komoditas
Tersisa      : 49 komoditas


In [20]:
# Profil kelengkapan dan status tiap komoditas
TOTAL_HARI = (df["tanggal"].max() - df["tanggal"].min()).days + 1
TANGGAL_AKHIR = df["tanggal"].max()
BATAS = 0.90          # syarat terisi 90%
MIN_HARI_BULAN = 15   # bulan dianggap valid kalau >= 15 hari berisi harga


def profil(g):
    g = g.sort_values("tanggal")
    awal, akhir = g["tanggal"].min(), g["tanggal"].max()
    hari_ada = g["tanggal"].nunique()
    hari_deret = (akhir - awal).days + 1
    bulan_deret = (akhir.to_period("M") - awal.to_period("M")).n + 1
    per_bulan = g.groupby("bulan")["tanggal"].nunique()
    lubang = g["tanggal"].drop_duplicates().diff().dt.days.dropna() - 1

    return pd.Series({
        "tanggal_awal": awal.date(),
        "tanggal_akhir": akhir.date(),
        "hari_tercatat": hari_ada,
        "pct_seluruh": round(hari_ada / TOTAL_HARI * 100, 1),     # terhadap seluruh periode
        "pct_dalam_deret": round(hari_ada / hari_deret * 100, 1), # terhadap deretnya sendiri
        "bulan_deret": bulan_deret,
        "bulan_valid": int((per_bulan >= MIN_HARI_BULAN).sum()),
        "lubang_terpanjang_hari": int(lubang.max()) if len(lubang) else 0,
        "masih_tercatat": akhir == TANGGAL_AKHIR,
    })


def klasifikasi(r):
    if r["komoditas"] in EXCLUDED:
        return "DIKECUALIKAN"
    if r["pct_seluruh"] >= BATAS * 100:
        return "LENGKAP"
    if r["bulan_deret"] < 13:
        return "FRAGMEN"
    if r["pct_dalam_deret"] >= BATAS * 100 and r["masih_tercatat"]:
        return "DERET PENDEK"
    return "BERLUBANG"


prof = df.groupby("komoditas").apply(profil).reset_index()
prof["status"] = prof.apply(klasifikasi, axis=1)

urutan = {"LENGKAP": 0, "DERET PENDEK": 1, "BERLUBANG": 2, "FRAGMEN": 3, "DIKECUALIKAN": 4}
prof = (prof.sort_values(by=["status", "pct_seluruh"], key=lambda s: s.map(urutan) if s.name == "status" else -s)
            .reset_index(drop=True))
prof.index = prof.index + 1

Path("../reports/tables").mkdir(parents=True, exist_ok=True)
prof.to_csv("../reports/tables/profil_kelengkapan.csv", index_label="no")

print(f"Total hari periode: {TOTAL_HARI:,}\n")
print(prof["status"].value_counts().reindex(urutan.keys()).to_string())
prof

Total hari periode: 1,339

status
LENGKAP         37
DERET PENDEK     7
BERLUBANG        3
FRAGMEN          2
DIKECUALIKAN    22


,komoditas,tanggal_awal,tanggal_akhir,hari_tercatat,pct_seluruh,pct_dalam_deret,bulan_deret,bulan_valid,lubang_terpanjang_hari,masih_tercatat,status
1,Bawang Bombay,2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
2,Bayam (Jenis Sayur),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
3,Beras Jagung,2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
4,Beras Ketan (Jenis Siyem),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
5,Buncis,2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
6,Garam Halus (Cap Kapal),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
7,Gula Jawa/Merah (Bahan Kelapa),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
8,Ikan Teri Asin (Bukan Teri Medan),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
9,Indomie Goreng,2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP
10,Jagung Pipilan Kering (Jenis Besar),2023-01-01,2026-08-31,1335,99.7,99.7,44,44,3,True,LENGKAP


In [21]:
# ===== ANOMALI: LONJAKAN PENDEK YANG KEMBALI KE LEVEL SEMULA =====
RASIO_BAWAH, RASIO_ATAS = 0.5, 2.0
JENDELA = 15            # median bergulir terpusat, kira kira 7 hari ke kiri dan kanan
MAKS_HARI_PENDEK = 3    # lonjakan lebih lama dari ini dianggap perubahan nyata, bukan salah catat

df = df.sort_values(["komoditas", "tanggal"]).reset_index(drop=True)
df["median_lokal"] = df.groupby("komoditas")["harga"].transform(
    lambda s: s.rolling(JENDELA, center=True, min_periods=8).median()
)
df["rasio"] = df["harga"] / df["median_lokal"]
df["di_luar"] = (df["rasio"] < RASIO_BAWAH) | (df["rasio"] > RASIO_ATAS)

# Panjang tiap blok hari berurutan yang di luar batas
sebelumnya = df.groupby("komoditas")["di_luar"].shift()
jeda_tgl = df.groupby("komoditas")["tanggal"].diff().dt.days.fillna(1) > 1
blok = ((df["di_luar"] != sebelumnya) | jeda_tgl).cumsum()
df["panjang_blok"] = df.groupby(blok)["di_luar"].transform("size").where(df["di_luar"], 0)

df["anomali"] = df["di_luar"] & (df["panjang_blok"] <= MAKS_HARI_PENDEK)   # ditandai, tidak dihapus
df["lonjakan_panjang"] = df["di_luar"] & ~df["anomali"]                     # dipertahankan sebagai perubahan nyata

print(f"Anomali pendek (ditandai)        : {df['anomali'].sum():,} hari")
print(f"Lonjakan panjang (dipertahankan) : {df['lonjakan_panjang'].sum():,} hari")

detail = (df[df["di_luar"]]
          .assign(jenis=lambda d: d["anomali"].map({True: "pendek, ditandai", False: "panjang, dipertahankan"}))
          [["tanggal", "komoditas", "harga", "median_lokal", "rasio", "panjang_blok", "jenis"]]
          .round({"median_lokal": 0, "rasio": 2}))
detail.to_csv("../reports/tables/anomali_harian_v2.csv", index=False)
display(detail[detail["komoditas"] == "Tomat"])

Anomali pendek (ditandai)        : 12 hari
Lonjakan panjang (dipertahankan) : 0 hari


,tanggal,komoditas,harga,median_lokal,rasio,panjang_blok,jenis


In [22]:
ringkas = (
    df.groupby("komoditas")
    .agg(hari=("harga", "size"),
         hari_anomali=("anomali", "sum"),
         harga_median=("harga", "median"))
)
ringkas["persen_anomali"] = (ringkas["hari_anomali"] / ringkas["hari"] * 100).round(2)
ringkas["dikecualikan"] = ringkas.index.isin(EXCLUDED)
ringkas = ringkas.sort_values("hari_anomali", ascending=False)

print(f"Komoditas dengan anomali: {(ringkas['hari_anomali'] > 0).sum()} dari {len(ringkas)}")
ringkas[ringkas["hari_anomali"] > 0]

Komoditas dengan anomali: 7 dari 71


,hari,hari_anomali,harga_median,persen_anomali,dikecualikan
komoditas,,,,,
Cabe Merah Kecil-Rawit,1335,4,39571.428571,0.30,True
Buncis,1335,2,13666.666667,0.15,False
Tahu biasa,1248,2,7785.714286,0.16,False
Jagung Pipilan Kering (Jenis Besar),1335,1,10166.666670,0.07,False
Kubis,1335,1,8166.666667,0.07,False
Telur Ayam Kampung,1334,1,2821.428571,0.07,False
Tempe Kw III (campuran kulit),1245,1,3000.000000,0.08,False


In [23]:
anom = df[df["anomali"]].copy()

per_bulan = (
    anom.groupby("bulan")
    .agg(hari_anomali=("harga", "size"), komoditas_terdampak=("komoditas", "nunique"))
    .sort_values("hari_anomali", ascending=False)
)
print("Bulan dengan anomali terbanyak:")
display(per_bulan.head(10))

per_tanggal = (
    anom.groupby("tanggal")
    .agg(komoditas_terdampak=("komoditas", "nunique"),
         contoh=("komoditas", lambda s: ", ".join(sorted(s.unique())[:5])))
    .sort_values("komoditas_terdampak", ascending=False)
)
print("\nTanggal dengan komoditas terdampak terbanyak:")
display(per_tanggal.head(15))

Bulan dengan anomali terbanyak:


,hari_anomali,komoditas_terdampak
bulan,,
2023-12,2,1
2025-09,2,1
2023-08,1,1
2024-07,1,1
2024-11,1,1
2024-10,1,1
2025-04,1,1
2025-05,1,1
2025-07,1,1



Tanggal dengan komoditas terdampak terbanyak:


,komoditas_terdampak,contoh
tanggal,,
2023-08-11,1,Tempe Kw III (campuran kulit)
2023-12-16,1,Cabe Merah Kecil-Rawit
2023-12-17,1,Cabe Merah Kecil-Rawit
2024-07-15,1,Telur Ayam Kampung
2024-10-17,1,Cabe Merah Kecil-Rawit
2024-11-18,1,Jagung Pipilan Kering (Jenis Besar)
2025-04-30,1,Cabe Merah Kecil-Rawit
2025-05-30,1,Tahu biasa
2025-07-30,1,Tahu biasa


In [24]:
MIN_PCT_TERISI = 90.0
MIN_BULAN_DERET = 36
AMBANG_UBAH = 0.005
MIN_KEAKTIFAN = 25.0

base = df[~df["anomali"]]
rata_bulanan = base.groupby(["komoditas", "bulan"])["harga"].mean().unstack("komoditas")
rata_bulanan = rata_bulanan.reindex(
    pd.period_range(rata_bulanan.index.min(), rata_bulanan.index.max(), freq="M")
)

perubahan = (rata_bulanan / rata_bulanan.shift(1) - 1).abs()
keaktifan = ((perubahan >= AMBANG_UBAH).sum() / perubahan.notna().sum() * 100).round(1)

gerbang = prof.set_index("komoditas")[
    ["status", "pct_seluruh", "bulan_deret", "lubang_terpanjang_hari", "masih_tercatat"]
].copy()
gerbang["n_satuan"] = df.groupby("komoditas")["satuan"].nunique()
gerbang["keaktifan_pct"] = keaktifan

cek = pd.DataFrame({
    "bukan_dikecualikan": ~gerbang.index.isin(EXCLUDED),
    "terisi_90": gerbang["pct_seluruh"] >= MIN_PCT_TERISI,
    "deret_36_bulan": gerbang["bulan_deret"] >= MIN_BULAN_DERET,
    "satu_satuan": gerbang["n_satuan"] == 1,
    "tercatat_bulan_akhir": gerbang["masih_tercatat"].astype(bool),
    "tidak_konstan": gerbang["keaktifan_pct"] >= MIN_KEAKTIFAN,
}, index=gerbang.index)

gerbang = gerbang.join(cek)
gerbang["lolos"] = cek.all(axis=1)
print(f"Total: {len(gerbang)} | Lolos gerbang: {gerbang['lolos'].sum()}")
print((~cek).sum().to_string())

Total: 71 | Lolos gerbang: 31
bukan_dikecualikan      22
terisi_90               15
deret_36_bulan          11
satu_satuan              0
tercatat_bulan_akhir     4
tidak_konstan           10


### Seleksi Komoditas

In [26]:
kandidat = gerbang[gerbang["lolos"]].index.tolist()
TOTAL_BULAN = df["bulan"].nunique()

def episode_lubang(g):
    tgl = g["tanggal"].drop_duplicates().sort_values()
    jeda = tgl.diff().dt.days.dropna() - 1
    jeda = jeda[jeda > 0]
    return pd.Series({"n_lubang": len(jeda), "lubang_terpanjang": int(jeda.max()) if len(jeda) else 0})

d = df[df["komoditas"].isin(kandidat)]
lub = d.groupby("komoditas").apply(episode_lubang)
pr = prof.set_index("komoditas").loc[kandidat]

ind = pd.DataFrame(index=kandidat)
ind.index.name = "komoditas"
ind["pct_hari_terisi"] = pr["pct_seluruh"]
ind["pct_bulan_valid"] = (pr["bulan_valid"] / TOTAL_BULAN * 100).round(1)
ind["pct_anomali"] = (d.groupby("komoditas")["anomali"].mean() * 100).round(3)
ind["lubang_terpanjang"] = lub["lubang_terpanjang"]
ind["n_lubang"] = lub["n_lubang"]
ind["bulan_deret"] = pr["bulan_deret"]
ind["keaktifan_pct"] = gerbang.loc[kandidat, "keaktifan_pct"]
ind = ind.sort_index()
ind.head()

,pct_hari_terisi,pct_bulan_valid,pct_anomali,lubang_terpanjang,n_lubang,bulan_deret,keaktifan_pct
komoditas,,,,,,,
Bawang Bombay,99.7,100.0,0.00,3,2,44,90.7
Bayam (Jenis Sayur),99.7,100.0,0.00,3,2,44,97.7
Beras Jagung,99.7,100.0,0.00,3,2,44,48.8
Beras Ketan (Jenis Siyem),99.7,100.0,0.00,3,2,44,58.1
Buncis,99.7,100.0,0.15,3,2,44,97.7


In [ ]:
# ===== TAHAP 1: KELAYAKAN DATA — kelengkapan, kontinuitas, konsistensi, panjang periode =====
import numpy as np

JUMLAH_PILIH = 10

# Ambang kelayakan (K1 sampai K3). Gerbang di sel sebelumnya sudah menyaring:
# kelengkapan harian >= 90%, panjang deret >= 36 bulan, satu satuan, masih tercatat di bulan akhir, tidak konstan
MIN_BULAN_VALID_PCT = 90.0   # K1: % bulan valid (>= 15 hari berisi harga)
MAKS_ANOMALI_PCT = 1.0       # K2: % hari anomali pendek
MAKS_LUBANG_HARI = 30        # K3: lubang data terpanjang (kontinuitas)

kand = gerbang[gerbang["lolos"]].join(ind[["pct_bulan_valid", "pct_anomali", "n_lubang"]])
lolos_k = ((kand["pct_bulan_valid"] >= MIN_BULAN_VALID_PCT)
           & (kand["pct_anomali"] <= MAKS_ANOMALI_PCT)
           & (kand["lubang_terpanjang_hari"] <= MAKS_LUBANG_HARI))

print(f"Lolos gerbang awal      : {len(kand)} komoditas")
print(f"Lolos kelayakan K1 - K3 : {int(lolos_k.sum())} komoditas")
if (~lolos_k).any():
    print("Gugur:", list(kand.index[~lolos_k]))
kand = kand[lolos_k].copy()
assert len(kand) >= JUMLAH_PILIH, "Kandidat layak kurang dari jumlah yang akan dipilih"
kand[["pct_seluruh", "bulan_deret", "keaktifan_pct", "pct_anomali", "lubang_terpanjang_hari"]].sort_index()


Lolos gerbang awal      : 31 komoditas
Lolos kelayakan K1 - K3 : 31 komoditas


,pct_seluruh,bulan_deret,keaktifan_pct,pct_anomali,lubang_terpanjang_hari
komoditas,,,,,
Bawang Bombay,99.7,44,90.7,0.000,3
Bayam (Jenis Sayur),99.7,44,97.7,0.000,3
Beras Jagung,99.7,44,48.8,0.000,3
Beras Ketan (Jenis Siyem),99.7,44,58.1,0.000,3
Buncis,99.7,44,97.7,0.150,3
Garam Halus (Cap Kapal),99.7,44,53.5,0.000,3
Gas LPG Tabung (3 Kg.),99.6,44,25.6,0.000,3
Gula Jawa/Merah (Bahan Kelapa),99.7,44,72.1,0.000,3
Ikan Teri Asin (Bukan Teri Medan),99.7,44,72.1,0.000,3


In [28]:
import numpy as np

rb = rata_bulanan[kand.index]
ubah = rb.pct_change(fill_method=None) * 100

volat = pd.DataFrame(index=kand.index)
volat["volatilitas_pct"] = ubah.std(ddof=1)
volat["volatilitas_robust"] = 1.4826 * (ubah - ubah.median()).abs().median()
volat["ubah_maks_naik"] = ubah.max()
volat["ubah_maks_turun"] = ubah.min()

def tren_log(s):
    s = s.dropna()
    x = np.array([p.ordinal for p in s.index], dtype=float)
    y = np.log(s.values.astype(float))
    slope = np.polyfit(x, y, 1)[0]
    r2 = np.corrcoef(x, y)[0, 1] ** 2
    return pd.Series({"tren_pct_per_bulan": (np.exp(slope) - 1) * 100, "r2_tren": r2,
                      "ubah_total_pct": (s.iloc[-1] / s.iloc[0] - 1) * 100})

volat = volat.join(rb.apply(tren_log).T)
volat["arah_tren"] = np.where(volat["r2_tren"] < 0.30, "tanpa tren jelas",
                              np.where(volat["tren_pct_per_bulan"] > 0, "naik", "turun"))

kand = kand.drop(columns=[c for c in volat.columns if c in kand.columns]).join(volat)
kand[["volatilitas_pct", "volatilitas_robust", "tren_pct_per_bulan", "arah_tren", "keaktifan_pct"]] \
    .sort_values("volatilitas_pct", ascending=False).round(2).head()

,volatilitas_pct,volatilitas_robust,tren_pct_per_bulan,arah_tren,keaktifan_pct
komoditas,,,,,
Tomat,26.04,15.10,-0.05,tanpa tren jelas,100.0
Buncis,13.35,12.16,0.27,tanpa tren jelas,97.7
Bayam (Jenis Sayur),11.81,4.77,1.19,naik,97.7
Bawang Bombay,10.65,4.05,0.43,tanpa tren jelas,90.7
Wortel (Kualitas Lokal),8.43,5.49,0.55,naik,88.4


In [30]:
# ===== PENAPISAN POLA EKSTREM AKIBAT KESALAHAN PENCATATAN =====
AMBANG_LONJAK = 0.40
AMBANG_KEMBALI = 0.15
MAKS_HARI_LONJAKAN = 0

rb = rata_bulanan[kand.index]
prev, nxt = rb.shift(1), rb.shift(-1)
chg = rb / prev - 1
net = nxt / prev - 1
pantulan = (chg.abs() >= AMBANG_LONJAK) & (net.abs() <= AMBANG_KEMBALI)
ekstrem_menetap = (chg.abs() >= AMBANG_LONJAK) & ~pantulan

d = df[df["komoditas"].isin(kand.index)]
hari_lonjakan = d[d["lonjakan_panjang"]].groupby("komoditas").size().reindex(kand.index).fillna(0).astype(int)

cek_salah = pd.DataFrame({
    "n_pantulan_bulanan": pantulan.sum().astype(int),
    "n_ubah_ekstrem_menetap": ekstrem_menetap.sum().astype(int),
    "hari_lonjakan_panjang": hari_lonjakan,
})
cek_salah["curiga_salah_catat"] = (cek_salah["n_pantulan_bulanan"] > 0) | (cek_salah["hari_lonjakan_panjang"] > MAKS_HARI_LONJAKAN)
print(f"Dicurigai salah catat: {int(cek_salah['curiga_salah_catat'].sum())} dari {len(cek_salah)}")
cek_salah.loc[["Tomat"]]

Dicurigai salah catat: 0 dari 31


,n_pantulan_bulanan,n_ubah_ekstrem_menetap,hari_lonjakan_panjang,curiga_salah_catat
komoditas,,,,
Tomat,0,4,0,False


In [32]:
# ===== PEMILIHAN 1 KOMODITAS UNTUK PEMODELAN FORECASTING =====
from pathlib import Path

kand = kand.drop(columns=["n_pantulan_bulanan", "n_ubah_ekstrem_menetap",
                          "hari_lonjakan_panjang", "curiga_salah_catat"], errors="ignore")
kand = kand.join(cek_salah[["n_pantulan_bulanan", "n_ubah_ekstrem_menetap",
                            "hari_lonjakan_panjang", "curiga_salah_catat"]])

bersih = kand[~kand["curiga_salah_catat"]].copy()
bersih["peringkat_volatilitas"] = bersih["volatilitas_pct"].rank(ascending=False, method="first").astype(int)
bersih["peringkat_robust"] = bersih["volatilitas_robust"].rank(ascending=False, method="first").astype(int)
bersih = bersih.sort_values("peringkat_volatilitas")

KOMODITAS_TERPILIH = bersih.index[0]

# Corong seleksi (untuk laporan)
corong = pd.Series({
    "Seluruh komoditas": len(gerbang),
    "Setelah 22 dikecualikan": int(cek["bukan_dikecualikan"].sum()),
    "Lolos gerbang": int(gerbang["lolos"].sum()),
    "Lolos kelayakan K1-K3": int(lolos_k.sum()),
    "Bersih dari dugaan salah catat": len(bersih),
    "Terpilih (volatilitas tertinggi)": 1,
}, name="jumlah_komoditas")
display(corong.to_frame())

# 5 teratas: cek apakah pilihan stabil terhadap ukuran volatilitas yang berbeda
KOLOM = ["peringkat_volatilitas", "volatilitas_pct", "peringkat_robust", "volatilitas_robust",
         "tren_pct_per_bulan", "arah_tren", "keaktifan_pct", "pct_seluruh",
         "bulan_deret", "lubang_terpanjang_hari", "pct_anomali"]
display(bersih[KOLOM].head(5).round(2))

selisih = bersih["volatilitas_pct"].iloc[0] / bersih["volatilitas_pct"].iloc[1]
print(f"Terpilih: {KOMODITAS_TERPILIH} | {selisih:.2f}x lebih volatil dari peringkat 2 ({bersih.index[1]})")

# Simpan hanya komoditas terpilih (lubang tidak ditambal, itu tugas tahap preprocessing)
Path("../data/").mkdir(parents=True, exist_ok=True)
Path("../reports/tables").mkdir(parents=True, exist_ok=True)

terpilih_df = (df[df["komoditas"] == KOMODITAS_TERPILIH][["tanggal", "komoditas", "satuan", "harga", "anomali"]]
               .sort_values("tanggal").reset_index(drop=True))
terpilih_df.to_csv("../data/processed/tomat_harian.csv", index=False, encoding="utf-8-sig")
bersih[KOLOM].head(10).round(2).to_csv("../reports/tables/seleksi_kandidat_teratas.csv")
print(f"Tersimpan: {len(terpilih_df):,} baris")

,jumlah_komoditas
Seluruh komoditas,71
Setelah 22 dikecualikan,49
Lolos gerbang,31
Lolos kelayakan K1-K3,31
Bersih dari dugaan salah catat,31
Terpilih (volatilitas tertinggi),1


,peringkat_volatilitas,volatilitas_pct,peringkat_robust,volatilitas_robust,tren_pct_per_bulan,arah_tren,keaktifan_pct,pct_seluruh,bulan_deret,lubang_terpanjang_hari,pct_anomali
komoditas,,,,,,,,,,,
Tomat,1,26.04,1,15.10,-0.05,tanpa tren jelas,100.0,99.7,44,3,0.00
Buncis,2,13.35,2,12.16,0.27,tanpa tren jelas,97.7,99.7,44,3,0.15
Bayam (Jenis Sayur),3,11.81,5,4.77,1.19,naik,97.7,99.7,44,3,0.00
Bawang Bombay,4,10.65,6,4.05,0.43,tanpa tren jelas,90.7,99.7,44,3,0.00
Wortel (Kualitas Lokal),5,8.43,4,5.49,0.55,naik,88.4,99.7,44,3,0.00


Terpilih: Tomat | 1.95x lebih volatil dari peringkat 2 (Buncis)
Tersimpan: 1,335 baris


Dari 31 kandidat tersebut, Tomat dipilih karena memiliki volatilitas tertinggi, yaitu simpangan baku perubahan harga bulanan sebesar 26,04%. 
Volatilitas tinggi membuat perbedaan kinerja antara model baseline dan metode lain terlihat jelas, sehingga tomat menjadi kasus uji yang menarik bagi model peramalan.